# Stage 7 (Updated): MultiAssetPurgedKFold Cross-Validation

Demonstrates **MultiAssetPurgedKFold** on the 10-stock pooled dataset
(2,071 events across AAPL, AMZN, BAC, GOOGL, JNJ, JPM, MSFT, NVDA, UNH, XOM).

Key properties vs single-stock `PurgedKFold`:
- Splits the **time axis** (unique event dates), not the row index
- All stocks at the same event date go to the same fold → no cross-sectional alpha leakage
- **Purging**: removes pre-test train events whose `t1` overlaps the test window
- **Embargo**: removes the first `pct_embargo` fraction of unique dates after test end

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
import sys
import os

sys.path.append('../')
from src.cross_validation import MultiAssetPurgedKFold, PurgedKFold, cv_score

plt.style.use('seaborn-v0_8-darkgrid')

## 1 & 2. Load Pooled Dataset and Extract Components

In [ ]:
dataset = pd.read_parquet('../data/processed/pooled_modelling.parquet')

meta_cols  = {'label', 't1', 'weight', 'ticker'}
feat_cols  = [c for c in dataset.columns if c not in meta_cols]
X = dataset[feat_cols]
y = dataset['label']
sample_weight = dataset['weight']
t1 = dataset['t1']

print(f"X shape   : {X.shape}  ({sum(not c.startswith('alpha') for c in feat_cols)} TS + "
      f"{sum(c.startswith('alpha') for c in feat_cols)} alpha features)")
print(f"y dist    : {dict(y.value_counts().sort_index())}")
print(f"Tickers   : {sorted(dataset['ticker'].unique())}")
print(f"Date range: {dataset.index.min().date()} -> {dataset.index.max().date()}")

## 3. Instantiate MultiAssetPurgedKFold

In [ ]:
pkf = MultiAssetPurgedKFold(n_splits=5, t1=t1, pct_embargo=0.01)
print(f"Instantiated MultiAssetPurgedKFold: n_splits={pkf.n_splits}, pct_embargo={pkf.pct_embargo}")

# Inspect fold structure
for fold_i, (train_idx, test_idx) in enumerate(pkf.split(X, y)):
    test_dates = X.iloc[test_idx].index
    tickers_in_test = sorted(dataset.iloc[test_idx]['ticker'].unique())
    print(f"  Fold {fold_i}: train={len(train_idx):4d}  test={len(test_idx):3d}  "
          f"test=[{test_dates.min().date()},{test_dates.max().date()}]  "
          f"tickers={tickers_in_test}")

## 4 & 5. Run Purged CV with RandomForest


In [ ]:
clf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=4,
                             class_weight='balanced')

purged_scores = cv_score(clf, X, y, sample_weight, scoring='accuracy', cv=pkf)
print("MultiAssetPurgedKFold CV Accuracy Scores:")
print(purged_scores.round(4).to_string())
print(f"\nMean Accuracy : {purged_scores.mean():.4f}")
print(f"Std Accuracy  : {purged_scores.std():.4f}")

## 6. Compare with Naive KFold


In [ ]:
naive_kf = KFold(n_splits=5, shuffle=False)
naive_scores = cv_score(clf, X, y, sample_weight=None, scoring='accuracy',
                        cv=naive_kf, t1=t1)

print("Naive KFold CV (no purge, no weights):")
print(naive_scores.round(4).to_string())
print(f"\nMean Accuracy  : {naive_scores.mean():.4f}")
print(f"Std Accuracy   : {naive_scores.std():.4f}")
print(f"Inflation (Naive - Purged): {naive_scores.mean() - purged_scores.mean():.4f}")
print()
print("NOTE: Naive KFold on a time-series violates temporal ordering.")
print("      Positive inflation indicates the test-set contains information")
print("      the model should not have had — a form of look-ahead leakage.")

## 7. Visualize Fold Composition


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Top: fold composition by time (all stocks stacked)
colors = plt.cm.tab10.colors
tickers = sorted(dataset['ticker'].unique())
ticker_color = {t: colors[i] for i, t in enumerate(tickers)}

for fold_i, (train_idx, test_idx) in enumerate(pkf.split(X, y)):
    test_dates = X.iloc[test_idx].index
    train_dates = X.iloc[train_idx].index
    axes[0].scatter(train_dates, [fold_i] * len(train_idx),
                    color='steelblue', s=8, alpha=0.3, label='Train' if fold_i == 0 else '')
    axes[0].scatter(test_dates, [fold_i] * len(test_idx),
                    color='crimson', s=8, alpha=0.8, label='Test' if fold_i == 0 else '')

axes[0].set_yticks(range(5))
axes[0].set_yticklabels([f'Fold {i}' for i in range(5)])
axes[0].set_xlabel('Event Date')
axes[0].set_title('MultiAssetPurgedKFold Composition — All Stocks')
axes[0].legend(loc='upper left')

# Bottom: accuracy comparison
folds = list(range(5))
axes[1].plot(folds, purged_scores.values, 'o-', color='steelblue', label=f'MultiAsset Purged (mean={purged_scores.mean():.3f})')
axes[1].plot(folds, naive_scores.values, 's--', color='grey', label=f'Naive KFold (mean={naive_scores.mean():.3f})')
axes[1].axhline(0.5, color='black', linestyle=':', alpha=0.5, label='Random baseline (0.50)')
axes[1].set_xticks(folds)
axes[1].set_xticklabels([f'Fold {i}' for i in folds])
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0.3, 0.8)
axes[1].set_title('Purged vs Naive CV Accuracy per Fold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/phase9_purged_vs_naive_cv.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Save Results


In [ ]:
results = pd.DataFrame({
    'MultiAsset_Purged_CV': purged_scores.values,
    'Naive_KFold':          naive_scores.values,
}, index=[f'fold_{i}' for i in range(5)])
results.loc['mean'] = results.mean()
results.loc['std']  = results.std().drop('mean')
results = results.round(4)
results.to_parquet('../data/processed/cv_multiasset_results.parquet')
print("Saved CV results to cv_multiasset_results.parquet")
print(results.to_string())